# Uchanganuzi wa Dai la Gharama

Daftari hii inaonyesha jinsi ya kuunda mawakala ambao hutumia viendelezi kuchakata gharama za safari kutoka kwa picha za risiti za eneo, kuunda barua pepe ya dai la gharama, na kuona data za gharama kwa kutumia chati ya pai. Mawakala huamua kazi kulingana na muktadha wa kazi.

Hatua:
1. Mwakala wa OCR anachakata picha ya risiti ya eneo na kutoa data za gharama za safari.
2. Mwakala wa Barua Pepe huunda barua pepe ya dai la gharama.

### Mfano wa hali ya gharama za safari:
Fikiria wewe ni mfanyakazi anayesafiri kwa mkutano wa biashara katika mji mwingine. Kampuni yako ina sera ya kurudisha gharama zote za kusafiri zinazofaa. Hapa kuna muhtasari wa gharama zinazowezekana za safari:
- Usafiri:
Tiketi za ndege kwa safari ya mzunguko kutoka mjini kwako kwenda mji wa kwenda.
Huduma za teksi au usafiri wa ride-hailing kwenda na kutoka uwanja wa ndege.
Usafiri wa ndani katika mji wa kwenda (kama usafiri wa umma, magari ya kukodi, au teksi).

- Malazi:
Malazi ya hoteli kwa usiku tatu katika hoteli ya biashara ya kiwango cha kati karibu na mahali pa mkutano.

- Vyakula:
Kiwango cha chakula cha kila siku kwa kifungua kinywa, chakula cha mchana, na chakula cha jioni, kulingana na sera ya kampuni ya per diem.

- Gharama Mbalimbali:
Ada za kuegesha gari uwanja wa ndege.
Gharama za kupata mtandao wa intaneti hoteli.
Tipsi au ada ndogo za huduma.

- Nyaraka:
Unawasilisha risiti zote (ndege, teksi, hoteli, chakula, n.k.) na ripoti kamili ya gharama kwa ajili ya kurudishiwa.


## Ingiza maktaba zinazohitajika

Ingiza maktaba na moduli zinazohitajika kwa daftari.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Tambua Mifano ya Matumizi

 Unda mfano wa Pydantic kwa ajili ya matumizi binafsi na darasa la ExpenseFormatter kubadilisha swali la mtumiaji kuwa data ya matumizi yenye muundo.

 Kila matumizi yataonyeshwa kwa muundo huu:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Kuelezwa Vyombo - Kutengeneza Barua Pepe

Unda kazi ya chombo ya kutengeneza barua pepe ya kuwasilisha dai la matumizi.
- Chombo hiki kinatumia dekoreta ya `@tool` kutoka kwenye Mfumo wa Wakala wa Microsoft.
- Hukokotoa jumla ya kiasi cha matumizi na kuunda maelezo katika mwili wa barua pepe.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Chombo cha Kutoa Gharama za Safari kutoka Picha za Risiti

Unda kazi ya chombo kutoa gharama za safari kutoka picha za risiti.
- Chombo hiki kinatumia `@tool` mtindo kutoka kwa Microsoft Agent Framework.
- Inasoma picha ya risiti, kuibadilisha kuwa base64, na kurudisha data URI kwa ajili ya wakala kuchambua.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Kuchakata Gharama

Tafsiri mawakala na waunganishe katika mtiririko wa kazi wa mfululizo kwa kutumia `WorkflowBuilder`.
- Wakala wa OCR hutambua data ya gharama iliyopangwa kutoka kwa picha ya risiti kwa kutumia chombo cha `load_receipt_image`.
- Wakala wa Barua Pepe huchukua data zilizotambuliwa na kuunda barua pepe rasmi ya dai la gharama kwa kutumia chombo cha `generate_expense_email`.
- `WorkflowBuilder` na `add_edge` huunda mfululizo wa mtiririko: Wakala wa OCR → Wakala wa Barua Pepe.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Kazi kuu

Jenga mtiririko wa kazi wa mfululizo na uukimbie kuchakata picha ya risiti na kuunda barua pepe ya madai ya gharama.


> **Kumbuka:** Mchakato huu kwa sasa unapitisha picha ya risiti kama maandishi ya base64, ambayo mifano mingi ya mazungumzo (ikiwa ni pamoja na gpt-4.1-mini) haitazingatia kama picha.
> Pia inawezekana kuvuka dirisha la muktadha la mfano. Inapendekezwa kutumia OCR na Azure AI Vision (au chombo kingine cha OCR) na kupitisha tu maandishi yaliyotolewa, au kubadilisha ili kutuma picha kama ujumbe wa `image_url`.
> Ikiwa unataka tu kuepuka makosa ya muktadha, jaribu picha ndogo ya risiti au mfano wenye dirisha kubwa zaidi la muktadha.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
